In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║         REMOVATION.AI — GOOGLE COLAB SETUP              ║
# ║  Run this ONE cell. Everything happens automatically.   ║
# ╚══════════════════════════════════════════════════════════╝
#
# FIRST RUN:  ~25 minutes (downloads models to Google Drive)
# AFTER THAT: ~5 minutes  (loads from Drive cache)

from google.colab import drive
import os, subprocess, sys, re, time, threading, requests, shutil

# ── CONFIG ──────────────────────────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

# ── STEP 1: Mount Drive ──────────────────────────────────────────────────────
print('━'*55)
print('STEP 1/8  Mount Google Drive')
print('━'*55)
drive.mount('/content/drive')
for d in [DRIVE_BASE, DRIVE_MODELS, DRIVE_NODES]:
    os.makedirs(d, exist_ok=True)
for s in MODEL_SUBDIRS:
    os.makedirs(f'{DRIVE_MODELS}/{s}', exist_ok=True)
print('✅ Drive mounted — storage:', DRIVE_BASE)

# ── STEP 2: Check GPU ────────────────────────────────────────────────────────
print()
print('━'*55)
print('STEP 2/8  GPU Check')
print('━'*55)
r = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
if r.returncode == 0:
    name, total = [x.strip() for x in r.stdout.strip().split(',')]
    print(f'✅ GPU: {name}  |  VRAM: {int(total)/1024:.1f} GB')
    if int(total) < 10000:
        print('⚠️  Less than 10GB — upgrade to T4 GPU in Runtime settings')
else:
    print('❌ No GPU detected — Runtime → Change runtime type → T4 GPU')
    sys.exit(1)

# ── STEP 3: Install ComfyUI ──────────────────────────────────────────────────
print()
print('━'*55)
print('STEP 3/8  Install ComfyUI')
print('━'*55)
if os.path.exists(f'{DRIVE_COMFY}/main.py'):
    print('Cached on Drive — copying to /content...')
    if os.path.exists(COMFYUI_PATH):
        os.system(f'rm -rf {COMFYUI_PATH}')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI ready from cache')
else:
    print('First time — cloning ComfyUI (~2 min)...')
    os.system(f'git clone https://github.com/comfyanonymous/ComfyUI.git {DRIVE_COMFY} -q')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI cloned and cached to Drive')

os.system(f'pip install -q -r {COMFYUI_PATH}/requirements.txt')

# Symlink all model dirs: Drive → ComfyUI
comfy_models = f'{COMFYUI_PATH}/models'
os.makedirs(comfy_models, exist_ok=True)
for subdir in MODEL_SUBDIRS:
    src  = f'{DRIVE_MODELS}/{subdir}'
    link = f'{comfy_models}/{subdir}'
    os.makedirs(src, exist_ok=True)
    if not os.path.exists(link):
        os.symlink(src, link)
print('✅ All model dirs linked from Drive')

# ── STEP 4: Fix model paths (runs every session) ─────────────────────────────
print()
print('━'*55)
print('STEP 4/8  Fix model paths')
print('━'*55)
sam2_src  = f'{DRIVE_MODELS}/sam2/sam2_hiera_small.pt'
gdino_src = f'{DRIVE_MODELS}/grounding-dino/groundingdino_swint_ogc.pth'
depth_src = f'{DRIVE_MODELS}/depth_anything_v2/depth_anything_v2_vits.pth'

# SAM2 — Impact Pack needs it in models/sams/ AND its own folder
for t in [
    f'{COMFYUI_PATH}/models/sams/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-impact-pack/models/sam2/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-segment-anything/models/sam2/sam2_hiera_small.pt',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(sam2_src):
        shutil.copy2(sam2_src, t)
print('✅ SAM2 paths')

# GroundingDINO
for t in [
    f'{NODES_PATH}/comfyui-segment-anything/models/groundingdino/groundingdino_swint_ogc.pth',
    f'{NODES_PATH}/comfyui-impact-pack/models/grounding-dino/groundingdino_swint_ogc.pth',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(gdino_src):
        shutil.copy2(gdino_src, t)
print('✅ GroundingDINO paths')

# Depth Anything
depth_target = f'{NODES_PATH}/comfyui_controlnet_aux/ckpts/depth_anything_v2_vits.pth'
os.makedirs(os.path.dirname(depth_target), exist_ok=True)
if not os.path.exists(depth_target) and os.path.exists(depth_src):
    shutil.copy2(depth_src, depth_target)
print('✅ Depth Anything path')

# CLIPSeg — git clone is blocked on Colab, use zip download
clipseg_path = f'{NODES_PATH}/ComfyUI-CLIPSeg'
if not os.path.exists(clipseg_path):
    print('⏬ Installing CLIPSeg...')
    os.system('wget -q -O /tmp/clipseg.zip https://github.com/biegert/ComfyUI-CLIPSeg/archive/refs/heads/main.zip')
    os.system('unzip -q /tmp/clipseg.zip -d /tmp/')
    os.system(f'cp -r /tmp/ComfyUI-CLIPSeg-main {clipseg_path}')
    os.system('pip install -q transformers')
    req = f'{clipseg_path}/requirements.txt'
    if os.path.exists(req):
        os.system(f'pip install -q -r {req} 2>/dev/null || true')
    print('✅ CLIPSeg installed')
else:
    print('✅ CLIPSeg already present')

# LaMa
lama_node_dir = f'{NODES_PATH}/comfyui-lama-remover/models'
lama_drive    = f'{DRIVE_NODES}/comfyui-lama-remover/models/big-lama.pt'
lama_target   = f'{lama_node_dir}/big-lama.pt'
os.makedirs(lama_node_dir, exist_ok=True)
if not os.path.exists(lama_target):
    if os.path.exists(lama_drive):
        shutil.copy2(lama_drive, lama_target)
        print('✅ LaMa copied from Drive')
    else:
        print('⏬ Downloading LaMa...', end=' ', flush=True)
        os.system(f'wget -q -c -O "{lama_drive}" "https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.pt"')
        if os.path.exists(lama_drive):
            shutil.copy2(lama_drive, lama_target)
            print('✅')
        else:
            print('❌')
else:
    print('✅ LaMa ready')

# ── STEP 5: Install custom nodes ─────────────────────────────────────────────
print()
print('━'*55)
print('STEP 5/8  Install custom nodes')
print('━'*55)
os.makedirs(NODES_PATH, exist_ok=True)

REQUIRED_NODES = [
    ('ComfyUI-Manager',             'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('comfyui_controlnet_aux',      'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
    ('ComfyUI-Advanced-ControlNet', 'https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git'),
    ('comfyui-inpaint-nodes',       'https://github.com/Acly/comfyui-inpaint-nodes.git'),
    ('comfyui-lama-remover',        'https://github.com/Layer-norm/comfyui-lama-remover.git'),
    ('ComfyUI-IC-Light',            'https://github.com/huchenlei/ComfyUI-IC-Light.git'),
    ('comfyui-impact-pack',         'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git'),
    ('rgthree-comfy',               'https://github.com/rgthree/rgthree-comfy.git'),
    ('comfyui-kjnodes',             'https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI_IPAdapter_plus',      'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('was-node-suite-comfyui',      'https://github.com/WASasquatch/was-node-suite-comfyui.git'),
    ('ComfyUI-GGUF',                'https://github.com/city96/ComfyUI-GGUF.git'),
    ('ComfyUI-RMBG',                'https://github.com/1038lab/ComfyUI-RMBG.git'),
    ('comfyui-segment-anything',    'https://github.com/storyicon/comfyui_segment_anything.git'),
]

for name, url in REQUIRED_NODES:
    drive_node = f'{DRIVE_NODES}/{name}'
    comfy_node = f'{NODES_PATH}/{name}'
    if os.path.exists(comfy_node):
        print(f'  ✅ {name}')
        continue
    if os.path.exists(drive_node):
        if not os.path.islink(comfy_node):
            os.symlink(drive_node, comfy_node)
        print(f'  🔗 {name} (Drive)')
    else:
        print(f'  ⏬ {name}...', end=' ', flush=True)
        ret = os.system(f'git clone {url} {drive_node} -q')
        if ret == 0:
            if not os.path.islink(comfy_node):
                os.symlink(drive_node, comfy_node)
            print('✅')
        else:
            print('❌')
    req = f'{comfy_node}/requirements.txt'
    if os.path.exists(req):
        os.system(f'pip install -q -r {req} 2>/dev/null || true')

os.system('cd /content/ComfyUI/custom_nodes/comfyui-impact-pack && python install.py 2>/dev/null || true')
print('✅ All custom nodes ready')

# ── STEP 6: Download models ───────────────────────────────────────────────────
print()
print('━'*55)
print('STEP 6/8  Download models (once to Drive)')
print('━'*55)

def get_model(subdir, filename, url, label):
    drive_path = f'{DRIVE_MODELS}/{subdir}/{filename}'
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > 100_000:
        print(f'  ✅ {label}')
        return True
    print(f'  ⏬ {label}...', end=' ', flush=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    os.system(f'wget -q -c -O "{drive_path}" "{url}"')
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > 100_000:
        print('✅')
        return True
    print('❌')
    return False

get_model('checkpoints','sd_xl_base_1.0.safetensors',
    'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
    'SDXL Base 1.0')
get_model('checkpoints','photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    'photon_v1')
get_model('checkpoints','1_5photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    '1_5photon_v1')
get_model('checkpoints','juggernautXL_version6Rundiffusion.safetensors',
    'https://huggingface.co/RunDiffusion/Juggernaut-XL-v6/resolve/main/juggernautXL_version6Rundiffusion.safetensors',
    'JuggernautXL v6')
get_model('controlnet','control_v11p_sd15_canny.pth',
    'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth',
    'ControlNet Canny SD1.5')
get_model('controlnet','control_v11p_sd15_mlsd_fp16.safetensors',
    'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_mlsd.pth',
    'ControlNet MLSD SD1.5')
get_model('controlnet','controlnet-canny-sdxl.safetensors',
    'https://huggingface.co/xinsir/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors',
    'ControlNet Canny SDXL')
get_model('upscale_models','RealESRGAN_x4plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN x4plus')
get_model('upscale_models','RealESRGAN_x4.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'RealESRGAN x4')
get_model('IC-Light','iclight_sd15_fc.safetensors',
    'https://huggingface.co/huchenlei/IC-Light/resolve/main/iclight_sd15_fc.safetensors',
    'IC-Light fc')
get_model('sam2','sam2_hiera_small.pt',
    'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
    'SAM2 Hiera Small')
get_model('depth_anything_v2','depth_anything_v2_vits.pth',
    'https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth',
    'Depth Anything V2')
get_model('grounding-dino','groundingdino_swint_ogc.pth',
    'https://huggingface.co/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth',
    'GroundingDINO')
get_model('inpaint','fooocus_inpaint_head.pth',
    'https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/main/fooocus_inpaint_head.pth',
    'Fooocus Inpaint Head')
get_model('clip','clip_l.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
    'CLIP-L')
get_model('clip','t5xxl_fp8_e4m3fn.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
    'T5-XXL fp8')
get_model('clip_vision','sigclip_vision_patch14_384.safetensors',
    'https://huggingface.co/Comfy-Org/sigclip_vision_384/resolve/main/sigclip_vision_patch14_384.safetensors',
    'SigCLIP Vision')

# Gated model status
print()
print('  Gated models (upload to Drive once to unlock):')
GATED = [
    ('diffusion_models','flux1-dev-fp8.safetensors',              'Furniture Staging'),
    ('vae',             'ae.safetensors',                         'Furniture Staging'),
    ('style_models',    'flux1-redux-dev.safetensors',            'Furniture Staging'),
    ('loras',           'flux_realism_lora.safetensors',          'Furniture Staging'),
    ('diffusion_models','qwen_image_edit_2509_fp8_e4m3fn.safetensors','Material Changer'),
    ('clip',            'qwen_2.5_vl_7b_fp8_scaled.safetensors', 'Material Changer'),
    ('vae',             'qwen_image_vae.safetensors',             'Material Changer'),
    ('loras',           'Qwen-Edit-2509-Multiple-angles.safetensors','Material Changer'),
    ('unet',            'qwen-image-edit-2509.Q4_K_M.gguf',      'Material Changer'),
]
for subdir, fname, feature in GATED:
    p = f'{DRIVE_MODELS}/{subdir}/{fname}'
    ok = os.path.exists(p) and os.path.getsize(p) > 100_000
    print(f'  {"✅" if ok else "❌"} {fname}')

# ── STEP 7: Start ComfyUI ─────────────────────────────────────────────────────
print()
print('━'*55)
print('STEP 7/8  Start ComfyUI (takes 3-4 min)')
print('━'*55)
os.system('pkill -f "python main.py" 2>/dev/null')
time.sleep(3)

comfy_log = open('/content/comfyui.log', 'w')
subprocess.Popen(
    ['python','main.py','--listen','0.0.0.0','--port','8188',
     '--preview-method','none','--lowvram'],
    cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
)

print('Loading all custom nodes', end='', flush=True)
comfy_ready = False
for _ in range(100):
    time.sleep(3)
    print('.', end='', flush=True)
    try:
        if requests.get('http://localhost:8188/system_stats', timeout=2).status_code == 200:
            comfy_ready = True
            print(' ✅')
            break
    except:
        pass

if not comfy_ready:
    print(' ❌')
    print('ComfyUI failed to start. Last 20 lines of log:')
    os.system('tail -20 /content/comfyui.log')
    sys.exit(1)

# ── STEP 8: Start tunnel ──────────────────────────────────────────────────────
print()
print('━'*55)
print('STEP 8/8  Start Cloudflare tunnel')
print('━'*55)
if not os.path.exists('/content/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(2)
tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared','tunnel','--url','http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

tunnel_url = None
print('Connecting', end='', flush=True)
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except:
        pass

# Start keep-alive thread
def _ping():
    while True:
        try: requests.get('http://localhost:8188/system_stats', timeout=5)
        except: pass
        time.sleep(30)
threading.Thread(target=_ping, daemon=True).start()

# Verify tunnel
if tunnel_url:
    try:
        resp = requests.get(f'{tunnel_url}/system_stats', timeout=20)
        d = resp.json()
        dev = d.get('devices', [{}])[0]
        free  = dev.get('vram_free',  0) / 1024**3
        total = dev.get('vram_total', 0) / 1024**3
        print(f'✅ Tunnel verified — {free:.1f} GB free / {total:.1f} GB total')
    except:
        print('⚠️  Tunnel URL ready but not verified yet — paste it anyway')

    print(f"""
╔══════════════════════════════════════════════════════╗
║           REMOVATION.AI COLAB READY                  ║
╠══════════════════════════════════════════════════════╣
║  1. Copy this URL:                                   ║
║     {tunnel_url:<50} ║
║                                                      ║
║  2. Admin panel → Settings → GPU → Cloud URL         ║
║     Paste URL → Mode: cloud → Save                   ║
║                                                      ║
║  3. Submit any room redesign to test                 ║
╚══════════════════════════════════════════════════════╝
""")
else:
    print('❌ No tunnel URL. Check: !cat /content/tunnel.log')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RECONNECT CELL — run this if tunnel URL expires
# Takes ~30 sec if ComfyUI still running, ~4 min if not
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, subprocess, requests, time, re

COMFYUI_PATH = '/content/ComfyUI'

# Check ComfyUI
try:
    requests.get('http://localhost:8188/system_stats', timeout=5)
    print('✅ ComfyUI still running')
except:
    print('ComfyUI stopped — restarting...', end='', flush=True)
    os.system('pkill -f "python main.py" 2>/dev/null')
    time.sleep(3)
    comfy_log = open('/content/comfyui.log', 'w')
    subprocess.Popen(
        ['python','main.py','--listen','0.0.0.0','--port','8188',
         '--preview-method','none','--lowvram'],
        cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
    )
    for _ in range(100):
        time.sleep(3)
        print('.', end='', flush=True)
        try:
            if requests.get('http://localhost:8188/system_stats', timeout=2).status_code == 200:
                print(' ✅')
                break
        except: pass
    else:
        print(' ❌ — check: !cat /content/comfyui.log')

# Restart tunnel
os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(2)
if not os.path.exists('/content/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared','tunnel','--url','http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

print('Getting URL', end='', flush=True)
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except: pass

if tunnel_url:
    print(f"""
╔══════════════════════════════════════════════════════╗
║  NEW URL — paste in admin panel:                     ║
║  {tunnel_url:<50} ║
║  Settings → GPU → Cloud URL → Save                  ║
╚══════════════════════════════════════════════════════╝
""")